# Entrenamiento de YOLOv8 para deteccion de stock en estanterias

Este notebook se ejecuta **en Google Colab, en la nube**, por lo que no necesitas instalar nada en tu computadora. Solo necesitas un navegador y una cuenta de Google.

**Como abrirlo:** entra a https://colab.research.google.com , ve a `Archivo > Subir cuaderno` y selecciona este archivo (`entrenar_yolo.ipynb`).

**Pasos generales:**
1. Instalar `ultralytics`
2. Preparar el dataset (fotos de tus productos/estantes + etiquetas)
3. Entrenar el modelo YOLOv8
4. Descargar el modelo entrenado (`best.pt`) y colocarlo en la carpeta `modelos/` del proyecto local

## 1. Instalar dependencias

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO
print('Ultralytics instalado correctamente')

## 2. (Opcional) Conectar tu Google Drive

Util si tienes tus fotos de estantes guardadas en Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## 3. Preparar el dataset

YOLOv8 necesita una estructura de carpetas asi:

```
dataset/
  images/
    train/   (fotos de estantes para entrenar)
    val/     (fotos de estantes para validar)
  labels/
    train/   (archivos .txt con las cajas etiquetadas, formato YOLO)
    val/
```

Para **etiquetar** tus propias fotos (dibujar las cajas alrededor de cada producto) se recomienda usar una herramienta gratuita como Roboflow (https://roboflow.com) o CVAT, que exportan directamente en formato YOLOv8 y generan el archivo `data.yaml` por ti.

Sube la carpeta `dataset/` ya etiquetada a este entorno (panel izquierdo de Colab -> icono de carpeta -> subir), o monta tu Google Drive en el paso anterior.

In [ ]:
# Ejemplo de archivo data.yaml que debe acompanar a tu dataset.
# Ajusta 'names' con los nombres reales de tus productos.

contenido_data_yaml = '''
train: dataset/images/train
val: dataset/images/val

nc: 3
names: ['producto_1', 'producto_2', 'estante_vacio']
'''

with open('data.yaml', 'w') as f:
    f.write(contenido_data_yaml)

print(contenido_data_yaml)

## 4. Entrenar el modelo

Se parte del modelo `yolov8n.pt` (version "nano", la mas liviana y rapida) y se ajusta (fine-tuning) con tus propias fotos.

In [ ]:
modelo = YOLO('yolov8n.pt')

resultados_entrenamiento = modelo.train(
    data='data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='restock_yolo'
)

## 5. Validar el modelo entrenado

In [ ]:
metricas = modelo.val()
print(metricas)

## 6. Descargar el modelo entrenado (best.pt)

El mejor modelo se guarda automaticamente en `runs/detect/restock_yolo/weights/best.pt`. Ejecuta la siguiente celda para descargarlo a tu computadora.

In [ ]:
from google.colab import files
files.download('runs/detect/restock_yolo/weights/best.pt')

## 7. Siguiente paso

Copia el archivo `best.pt` descargado dentro de la carpeta `modelos/` de tu proyecto local (junto a `dashboard.py`, `config.py`, etc). El modulo `vision/deteccion_stock.py` lo detectara automaticamente y dejara de usar el modelo generico de demo.